In [ ]:
# -*- coding: utf-8 -*-
# 该脚本把BigQuant的表以Parquet数据库的形式保存在指定目录下，每个文件不超过32M,需要手动下载到本地。
import os
import dai
import datetime as dt
import pandas as pd

TABLE = "cn_stock_bar1d"
START_DATE = "2005-01-01"
END_DATE = dt.date.today().strftime("%Y-%m-%d")

OUT_DIR = "cn_stock_bar1d_parquet_bisect_32mb"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_BYTES = 32 * 1024 * 1024  # 32MB
COMPRESSION = "zstd"          # 不支持就改 "snappy"

def q(s, e):
    sql = f"""
    SELECT *
    FROM {TABLE}
    WHERE date >= '{s}' AND date <= '{e}'
    """
    return dai.query(sql).df()

def save_parquet(df, path):
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    df.to_parquet(path, index=False, compression=COMPRESSION)

def sizeof(path):
    return os.path.getsize(path) if os.path.exists(path) else 0

def tag(s_date, e_date):
    return f"{s_date.strftime('%Y%m%d')}_{e_date.strftime('%Y%m%d')}"

def export_range(s_date, e_date, depth=0, max_depth=60):
    """最少文件优先：能大就大；超限就二分。"""
    if depth > max_depth:
        raise RuntimeError("Max recursion depth reached, please narrow range or add further split logic.")

    s = s_date.strftime("%Y-%m-%d")
    e = e_date.strftime("%Y-%m-%d")
    t = tag(s_date, e_date)

    out_path = os.path.join(OUT_DIR, f"{TABLE}_{t}.parquet")

    df = q(s, e)
    if len(df) == 0:
        return 0  # no file

    save_parquet(df, out_path)
    sz = sizeof(out_path)

    if sz <= MAX_BYTES:
        print(f"[OK] {t} rows={len(df):,} size={sz/1024/1024:.2f}MB")
        return 1

    # 超限：删除该文件，二分区间
    os.remove(out_path)
    print(f"[SPLIT] {t} size={sz/1024/1024:.2f}MB > 32MB -> bisect")

    if s_date == e_date:
        # 极罕见：单日都超限（理论上不太可能），需要再按股票分段切
        raise RuntimeError(f"Single day {s_date} still >32MB, need split by instrument range.")

    mid = (pd.to_datetime(s_date) + (pd.to_datetime(e_date) - pd.to_datetime(s_date)) / 2).date()
    # 确保左右区间都有推进
    left_s, left_e = s_date, mid
    right_s, right_e = (mid + dt.timedelta(days=1)), e_date

    n1 = export_range(left_s, left_e, depth+1, max_depth)
    n2 = export_range(right_s, right_e, depth+1, max_depth)
    return n1 + n2

# 入口
s0 = pd.to_datetime(START_DATE).date()
e0 = pd.to_datetime(END_DATE).date()
nfiles = export_range(s0, e0)
print("Done. files:", nfiles)